# 01 - Legacy Tasic evidence boundary

- **Meeting item:** Tasic notebook conclusions
- **Commit:** C02 - `docs: preserve legacy Tasic evidence boundary`
- **Commit achievement:** converts the earlier Tasic discussion into a traceable evidence record with checksums.
- **Data mode:** `legacy read-only`. This notebook reads tracked output files only. It downloads nothing and writes nothing under `experiments/`.
- **Evidence labels used:** verified, experimental result, inference, proposed, unknown
- **Human gate:** can a reviewer separate verified result, experimental result, inference, proposed work, and unknowns?
- **Stop condition:** any condition-specific, independent-modality, or causal claim appears.

If the tracked legacy outputs are unavailable, the gate below becomes `BLOCKED` and nothing is inferred.

## 1. Manifest integrity (evidence label: verified)

Eight tracked files are pinned by byte size and SHA-256 in
`legacy/tasic_proxy_view/manifest.json`. If any of them changes, this cell fails.

In [1]:
import importlib.util
import json
import sys

import pandas as pd

import p22

REPO_ROOT = p22.REPO_ROOT


def load_module(relative_path: str, name: str):
    spec = importlib.util.spec_from_file_location(name, REPO_ROOT / relative_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


builder = load_module("scripts/build_legacy_manifest.py", "legacy_manifest")
manifest = builder.load_manifest()
inputs_available = all(
    (REPO_ROOT / entry["relative_path"]).is_file() for entry in manifest["files"]
)
print(f"legacy inputs available: {inputs_available}")
problems = builder.verify_manifest(manifest) if inputs_available else ["legacy inputs unavailable"]
print(f"manifest problems: {problems or 'none'}")
print(f"data mode: {manifest['data_mode']}")
pd.DataFrame(manifest["files"])[["relative_path", "bytes", "evidence_label", "generator"]]

legacy inputs available: True
manifest problems: none
data mode: legacy read-only


,relative_path,bytes,evidence_label,generator
0,experiments/run_tasic2018_pipeline.py,39140,verified,human-authored legacy script
1,experiments/tasic2018_summary.json,6481,experimental result,experiments/run_tasic2018_pipeline.py
2,experiments/tasic2018_broadtype_metrics.csv,851,experimental result,experiments/run_tasic2018_pipeline.py
3,experiments/tasic2018_finegrained_metrics.csv,1457,experimental result,experiments/run_tasic2018_pipeline.py
4,experiments/tasic2018_attn_broadtype.csv,205,experimental result,experiments/run_tasic2018_pipeline.py
5,experiments/tasic2018_attn_finegrained.csv,5320,experimental result,experiments/run_tasic2018_pipeline.py
6,experiments/tasic2018_mcnemar.json,234,experimental result,experiments/run_tasic2018_pipeline.py
7,experiments/tasic2018_mutual_information.json,261,experimental result,experiments/run_tasic2018_pipeline.py


## 2. What was run (evidence label: verified)

Both views come from one RNA count matrix. View one is highly variable gene expression;
view two is a principal component projection of the same matrix.

In [2]:
summary = json.loads((REPO_ROOT / "experiments/tasic2018_summary.json").read_text())
dataset = summary["dataset"]
for key, value in dataset.items():
    print(f"{key}: {value}")
print("")
print(f"four-class task classes: {summary['broad_type']['class_names']}")
print(f"fine-grained classes: {summary['fine_grained']['n_classes']}")
print(f"fine-grained train/val/test cells: {summary['fine_grained']['train_val_test']}")
print(f"single source matrix for both views: {True}")

source: Tasic et al. 2018, GEO GSE115746
total_cells_raw: 22113
total_cells_after_qc: 22113
total_genes_after_qc: 40333
hvg_count: 2000
pca_components: 60
pca_variance_explained: 0.3797

four-class task classes: ['Endothelial', 'GABAergic', 'Glutamatergic', 'Non-Neuronal']
fine-grained classes: 130
fine-grained train/val/test cells: [13988, 3498, 4372]
single source matrix for both views: True


## 3. Recorded numbers (evidence label: experimental result)

The four-class task is saturated, so it cannot separate model families. The fine-grained
task is the only one with usable spread, and it was run with one seed.

In [3]:
rows = []
for task in ("broad_type", "fine_grained"):
    for model, metrics in summary[task]["results"].items():
        rows.append(
            {
                "task": task,
                "model": model,
                "accuracy": round(metrics["accuracy"], 4),
                "f1_weighted": round(metrics["f1_weighted"], 4),
                "interval": f"[{metrics['accuracy_ci_lo']:.4f}, {metrics['accuracy_ci_hi']:.4f}]",
                "interval_unit": "test cells (not donors)",
                "seeds": 1,
            }
        )
pd.DataFrame(rows)

,task,model,accuracy,f1_weighted,interval,interval_unit,seeds
0,broad_type,LogReg_GeneExpr,1.0000,1.0000,"[1.0000, 1.0000]",test cells (not donors),1
1,broad_type,MLP_GeneExpr,1.0000,1.0000,"[1.0000, 1.0000]",test cells (not donors),1
2,broad_type,MLP_MorphProxy,0.9998,0.9998,"[0.9993, 1.0000]",test cells (not donors),1
3,broad_type,NaiveConcatFusion,1.0000,1.0000,"[1.0000, 1.0000]",test cells (not donors),1
4,broad_type,CrossAttentionFusion,1.0000,1.0000,"[1.0000, 1.0000]",test cells (not donors),1
5,fine_grained,LogReg_GeneExpr,0.9101,0.9090,"[0.9014, 0.9181]",test cells (not donors),1
6,fine_grained,MLP_GeneExpr,0.9030,0.9043,"[0.8941, 0.9113]",test cells (not donors),1
7,fine_grained,MLP_MorphProxy,0.9135,0.9146,"[0.9055, 0.9220]",test cells (not donors),1
8,fine_grained,NaiveConcatFusion,0.9213,0.9221,"[0.9133, 0.9289]",test cells (not donors),1
9,fine_grained,CrossAttentionFusion,0.9149,0.9165,"[0.9067, 0.9229]",test cells (not donors),1


In [4]:
mcnemar = summary["mcnemar"]
print("paired comparison, attention fusion against concatenation fusion, fine-grained task")
for key, value in mcnemar.items():
    print(f"  {key}: {value}")
print("")
print(
    "reading: accuracy difference of "
    f"{mcnemar['concat_acc'] - mcnemar['attn_acc']:.4f} in favour of concatenation, "
    f"p = {mcnemar['p_value']}, not significant at 0.05"
)

paired comparison, attention fusion against concatenation fusion, fine-grained task
  attn_correct_concat_wrong: 128
  attn_wrong_concat_correct: 156
  chi2_continuity_corrected: 2.5669
  p_value: 0.109121
  significant_005: False
  attn_acc: 0.9149130832570905
  concat_acc: 0.9213174748398902

reading: accuracy difference of 0.0064 in favour of concatenation, p = 0.109121, not significant at 0.05


In [5]:
broad_attn = pd.read_csv(REPO_ROOT / "experiments/tasic2018_attn_broadtype.csv")
fine_attn = pd.read_csv(REPO_ROOT / "experiments/tasic2018_attn_finegrained.csv")
mutual_information = summary["mutual_information"]

print("routing weights are signals, not explanations: no intervention test was run")
print("")
print("four-class task, mean weight on view one per class:")
print(broad_attn[["cell_type", "n_test", "genomic_mean", "program_mean"]].to_string(index=False))
print("")
print(f"fine-grained task classes: {len(fine_attn)}")
print(f"  mean weight on view one: {fine_attn['genomic_mean'].mean():.4f}")
print(f"  mean weight on view two: {fine_attn['program_mean'].mean():.4f}")
print("")
print("mutual information with the label:")
print(f"  view one features, mean: {mutual_information['hvg_to_label_mean']}")
print(f"  view two components, mean: {mutual_information['pca_to_label_mean']}")
print(f"  paired view features, mean: {mutual_information['pairwise_hvg_pca_mean']}")

routing weights are signals, not explanations: no intervention test was run

four-class task, mean weight on view one per class:
    cell_type  n_test  genomic_mean  program_mean
 Non-Neuronal     201        0.6078        0.3922
    GABAergic    1912        0.8474        0.1526
  Endothelial      62        0.8688        0.1312
Glutamatergic    2248        0.8949        0.1051

fine-grained task classes: 130
  mean weight on view one: 0.0947
  mean weight on view two: 0.9053

mutual information with the label:
  view one features, mean: 0.1132
  view two components, mean: 0.6033
  paired view features, mean: 0.0027


## 4. Inference, proposed work, and boundary

**Inference (supported as association only)**

- The fusion architecture runs end to end at this scale and emits per-class routing weights.
- Concatenation fusion is at least as accurate as attention fusion here; the paired test
  shows no significant difference.
- Asymmetric routing weights coincide with the view whose features carry higher mutual
  information with the label in this setup.

**Proposed (not yet done)**

- Donor-held-out splitting, train-only transforms, five-seed reporting, and held-out
  interventions on synthetic data: steps C03 to C11 of this plan.

The boundary list below is stored in the manifest, so a later reader cannot separate the
numbers from their limits.

In [6]:
print("boundary recorded in manifest:")
for item in manifest["boundary"]:
    print(f"  - {item}")
print("")
print("unknown:")
print("  - whether donor identifiers exist for these cells; the tracked outputs carry none")
print("  - whether the model ranking survives donor-held-out splits, train-only transforms, five seeds")
print("  - whether anything here transfers to a second measurement modality")

boundary recorded in manifest:
  - Both views are derived from one RNA count matrix: view one is highly variable gene expression, view two is a principal component projection of the same matrix.
  - This is an architecture proof of concept, not independent two-modality validation.
  - This is not condition-specific evidence and contains no clinical cohort.
  - Attention weights are routing signals; no intervention test supports treating them as explanations.
  - The projection for view two was fitted on all cells before splitting, so leakage cannot be ruled out.
  - Splits were stratified over cells with one seed; no donor grouping and no seed variability estimate.
  - Reported intervals were resampled over test cells, not over donors.

unknown:
  - whether donor identifiers exist for these cells; the tracked outputs carry none
  - whether the model ranking survives donor-held-out splits, train-only transforms, five seeds
  - whether anything here transfers to a second measurement moda

## 5. Gate checks

The last check scans the legacy outputs for the condition terms listed in
`plan/approvals.json`. Finding one would mean these outputs are not condition-free.

In [7]:
import re

assert inputs_available, "legacy inputs unavailable: gate is BLOCKED, infer nothing"
assert problems == [], problems
assert len(manifest["files"]) == 8
assert manifest["data_mode"] == p22.LEGACY_READ_ONLY
assert manifest["source_dataset"]["matrix_tracked_in_repository"] is False

terms = p22.load_approvals()["condition_terms"]
hits = []
for entry in manifest["files"]:
    if entry["relative_path"].endswith(".py"):
        continue
    text = (REPO_ROOT / entry["relative_path"]).read_text(errors="replace").lower()
    for term in terms:
        if re.search(rf"(?<![a-z0-9]){re.escape(term.lower())}(?![a-z0-9])", text):
            hits.append((entry["relative_path"], term))
assert hits == [], hits
print(f"scanned {len(manifest['files']) - 1} output files for {len(terms)} condition terms: no hits")
print("all C02 gate checks passed")

scanned 7 output files for 11 condition terms: no hits
all C02 gate checks passed


## Gate

**PASS**

- **Observed:** eight legacy files pinned by checksum; numbers separated by evidence label; the four-class task is saturated at 1.0; on the fine-grained task concatenation fusion is 0.9213 against attention fusion 0.9149 with p = 0.109; routing weights recorded as signals only; boundary stored in the manifest and in `legacy/tasic_proxy_view/README.md`.
- **Remains unknown:** whether donor identifiers exist for these cells, whether the ranking survives donor-held-out evaluation with train-only transforms and five seeds, and whether any of it transfers to a second measurement modality.
- **Next decision owner:** researcher continues with C03 on synthetic data. Reuse of these numbers outside this boundary needs the named approver.